# Gate detection — asymmetric close + fill + drop border filled contours

```
raw frame
  → threshold
  → MORPH_CLOSE (close_short, close_long) → bridges LEDs along vertical-ish sides
  → MORPH_CLOSE (close_long, close_short) → bridges LEDs along horizontal-ish sides
  → temporary extra dilation (only to seal gaps for flood fill)
  → floodFill from image corners → background = 255
  → invert                       → enclosed gate interiors = 255
  → OR with the closed rim       → solid filled gates
  → MORPH_OPEN  (~3 px)
  → findContours(RETR_EXTERNAL)
  → drop contours whose bbox touches the image border
  → drop small contours
  → pick largest survivor → one contour ≈ one gate
  → smooth + curvature peaks → 4 corners
  → cv2.undistortPoints → normalized coords
```

In [ ]:
import sys
sys.path.insert(0, '../myassignement')

from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

from cv_detection import (
    K, DIST,
    preprocess_fill, detect_gate, render_detection,
)

In [ ]:
folder = Path('../myassignement/frames')
imgs = sorted(
    list(folder.rglob('*.jpg')) + list(folder.rglob('*.png')),
    key=lambda p: p.stat().st_mtime,
    reverse=True,   # newest first → idx=0 is always the latest capture
)
print(f'Found {len(imgs)} images in {folder} (newest first)')

## Preprocessing stages

In [ ]:
idx = 0
gray = cv2.imread(str(imgs[idx]), cv2.IMREAD_GRAYSCALE)
p = preprocess_fill(gray)

stages = [
    (gray,           'grayscale'),
    (p['raw'],       'threshold'),
    (p['closed'],    '2× asymm close'),
    (p['seal'],      'seal dilation'),
    (p['interior'],  'flood-fill interior'),
    (p['filled'],    'filled gate'),
]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (img, title) in zip(axes.flat, stages):
    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
fig.suptitle(f'Preprocessing stages — {imgs[idx].name}', fontsize=11)
plt.tight_layout()
plt.show()

## Single-frame inspection

In [ ]:
idx = 0
gray = cv2.imread(str(imgs[idx]), cv2.IMREAD_GRAYSCALE)
result = detect_gate(gray, K, DIST)
prep   = result['prep']

stages = [
    (gray,              'raw'),
    (prep['closed'],    '2× asymm close'),
    (prep['seal'],      'seal dilation'),
    (prep['filled'],    'filled gate'),
    (render_detection(gray, result), f'detection  [{result["status"]}]'),
]
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (img, title) in zip(axes, stages):
    kw = {} if img.ndim == 3 else {'cmap': 'gray', 'vmin': 0, 'vmax': 255}
    ax.imshow(img, **kw)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
fig.suptitle(imgs[idx].name, fontsize=10)
plt.tight_layout()
plt.show()

if result['curvature'] is not None:
    fig, ax = plt.subplots(figsize=(11, 2.5))
    ax.plot(result['curvature'], color='steelblue', lw=1)
    ax.set_ylabel('curvature (rad)')
    ax.set_xlabel('contour point index')
    if result['quad_pix'] is not None and result['smoothed'] is not None:
        sm = result['smoothed']
        for i, q in enumerate(result['quad_pix']):
            idx_c = int(np.argmin(np.linalg.norm(sm - q, axis=1)))
            ax.axvline(idx_c, color='r', alpha=0.7, lw=1.5)
            ax.text(idx_c, ax.get_ylim()[1] * 0.9, f'C{i}',
                    ha='center', fontsize=8, color='r')
    ax.set_title('curvature along the filled-gate contour')
    plt.tight_layout()
    plt.show()

if result['quad_norm'] is not None:
    print('Normalised corners (undistorted):')
    for lbl, pt in zip(['TL', 'TR', 'BR', 'BL'], result['quad_norm']):
        print(f'  {lbl}: ({pt[0]:+.4f},  {pt[1]:+.4f})')

## Grid sweep on half the images

In [ ]:
subset = imgs[::2]
cols = 5
rows = int(np.ceil(len(subset) / cols))

STATUS_COLOR = {
    'ok':              '#00cc44',
    'no_gate':         '#cc3300',
    'no_corners':      '#ff9900',
    'commit_to_pass':  '#cc44ff',
}

fig, ax = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 2.8))
ax = np.atleast_2d(ax)
for k, p in enumerate(subset):
    r, c = divmod(k, cols)
    g   = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    res = detect_gate(g, K, DIST)
    ax[r, c].imshow(render_detection(g, res))
    status = res['status']
    color  = STATUS_COLOR.get(status, 'white')
    ax[r, c].set_title(f'{p.name}\n{status}', fontsize=6,
                        color=color, fontweight='bold')
    ax[r, c].axis('off')
    for spine in ax[r, c].spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)
        spine.set_visible(True)

for k in range(len(subset), rows * cols):
    r, c = divmod(k, cols)
    ax[r, c].axis('off')

fig.suptitle(
    f'Grid sweep — {len(subset)} frames   '
    '[green=ok  orange=no_corners  red=no_gate  violet=commit_to_pass]',
    fontsize=9)
plt.tight_layout()
plt.show()